# Holiday Identification by Daily Profile Clustering

This notebook flags atypical days in one demand series by comparing each daily profile against its `(segment, weekday)` reference group.

It loads observed data, builds day-level profiles, measures distance to the group centroid, and compares detected outliers against the local holiday catalog.

This is exploratory notebook code, not part of the production pipeline.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analog_holidays.dataset_config import ACTIVE_CONFIG, list_dataset_regions
from analog_holidays.identify_HOLIDAYS import (
    build_holiday_groups,
    build_wide_df,
    classify_holiday_weekend_type,
    cluster_atypical_profiles,
    compare_outliers_holidays,
    compute_distances,
    detect_outliers,
    find_holidays_not_detected,
    get_date_sets,
    get_hour_cols,
    load_holidays_catalog,
    load_results_data,
    plot_cluster_atypical,
    plot_distance_distribution,
    plot_profiles_by_holiday,
    plot_profiles_by_segment_dow,
    print_summary,
    report_nth_monday_holidays,
 )

DEMAND_PATH = ACTIVE_CONFIG.demand_path
HOLIDAYS_PATH = ACTIVE_CONFIG.notebook_holidays_path

print(f'Active dataset: {ACTIVE_CONFIG.key}')
print(f'Demand CSV: {DEMAND_PATH}')
print(f'Holidays: {HOLIDAYS_PATH}')

## 1. Configuration

In [ ]:
AVAILABLE_UNIQUE_IDS = list_dataset_regions()
if not AVAILABLE_UNIQUE_IDS:
    raise ValueError(f'No configured series were found in {DEMAND_PATH}.')

UNIQUE_ID = AVAILABLE_UNIQUE_IDS[0]
DATE_END = '2025-01-01'

MONTH_NAMES = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December',
]

SEGMENTS = [
    {'label': 'Spring', 'months': [3, 4, 5]},
    {'label': 'Summer', 'months': [6, 7, 8]},
    {'label': 'Autumn', 'months': [9, 10, 11]},
    {'label': 'Winter', 'months': [12, 1, 2]},
]

SEGMENTS = [{'label': month_name, 'months': [month_number]} for month_number, month_name in enumerate(MONTH_NAMES, start=1)]

OUTLIER_PERCENTILE = 95
EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE = True
KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE = 75
DISTANCE_METRIC = 'PEARSON'

WEEKDAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

## 2. Load observed data from CSV

In [ ]:
MONTH_TO_SEGMENT = {month: segment['label'] for segment in SEGMENTS for month in segment['months']}
SEGMENT_LABELS = [segment['label'] for segment in SEGMENTS]

df_raw = load_results_data(DEMAND_PATH, UNIQUE_ID)
if DATE_END is not None:
    cutoff_ts = pd.Timestamp(DATE_END)
    df_raw = df_raw[df_raw['ds'] < cutoff_ts].copy()
    if df_raw.empty:
        raise ValueError(f'No data available for {UNIQUE_ID} before {cutoff_ts.date()}.')

print(f'Series: {UNIQUE_ID}')
print(f'Records: {len(df_raw):,}')
if DATE_END is not None:
    print(f'DATE_END cutoff: {pd.Timestamp(DATE_END).date()} (exclusive)')
print(f'Range: {df_raw["ds"].min()} → {df_raw["ds"].max()}')
df_raw.head()

## 3. Pivot to wide format (1 row = 1 day, 24 columns = hours)

In [ ]:
df_wide = build_wide_df(df_raw, MONTH_TO_SEGMENT, exclude_years=[2022])
HOUR_COLS = get_hour_cols(df_wide)
year_min = int(df_wide.index.year.min())
year_max = int(df_wide.index.year.max())
df_holidays = load_holidays_catalog(HOLIDAYS_PATH, year_min, year_max)
df_holidays_display = df_holidays.copy()
KNOWN_HOLIDAY_DATES = (
    set(pd.to_datetime(df_holidays['date']).dt.normalize())
    if EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE else set()
 )

print(f'Complete days: {len(df_wide)}')
print(f'Segments ({len(SEGMENT_LABELS)}):')
for segment_label in SEGMENT_LABELS:
    n_days = (df_wide['segment'] == segment_label).sum()
    print(f'  {segment_label}: {n_days} days')
print(f'Catalog holidays in range: {len(df_holidays)}')
print(f'Dates excluded from the baseline: {len(KNOWN_HOLIDAY_DATES)}')
df_wide.head()

## 4. Compute distances by segment and weekday

Distances are computed inside each `(segment, dow)` group against that group's reference centroid.

`DISTANCE_METRIC` controls whether the score emphasizes magnitude, shape, or both.

In [ ]:
df_dist = compute_distances(
    df_wide,
    HOUR_COLS,
    SEGMENT_LABELS,
    WEEKDAY_NAMES,
    distance_metric=DISTANCE_METRIC,
    reference_exclude_dates=KNOWN_HOLIDAY_DATES,
 )

print(f'Distance metric: {DISTANCE_METRIC}')
if DISTANCE_METRIC == 'PEARSON_EUCLIDIAN':
    print('  Euclidean and Pearson components combined after group-wise normalization')
elif DISTANCE_METRIC == 'EUCLIDIAN':
    print('  Euclidean distance only')
elif DISTANCE_METRIC == 'PEARSON':
    print('  1 - Pearson r only')
print(f'\nSegments: {SEGMENT_LABELS}')
print(f'Total records (day × group): {len(df_dist)}')
df_dist.head(10)

## 5. Identify outliers (days far from their group centroid)

In [ ]:
df_dist, df_outliers = detect_outliers(
    df_dist,
    OUTLIER_PERCENTILE,
    threshold_reference_only=EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE,
    promote_dates=KNOWN_HOLIDAY_DATES,
    promote_min_group_percentile=KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE,
)

print(f'Detected outliers: {len(df_outliers)}')
print(f'Out of a total of {len(df_dist)} days')
print(f'Percentage: {100 * len(df_outliers) / len(df_dist):.1f}%')
print(f'Known holidays rescued by within-group percentile: {int(df_dist["is_promoted_outlier"].sum())}')
df_outliers.head(20)

## 6. Load the local recognized holidays catalog (`holidays_recognized.json`)

In [ ]:
print(f'Generated holidays: {len(df_holidays_display)} (from {year_min} to {year_max})')
df_holidays_display.tail(15)


## 6b. Holidays with an nth-Monday rule

Three civic holidays moved from fixed dates to observed Mondays after the 2006 Federal Labor Law reform. Years before 2006 keep the historical fixed date.

| Holiday | Before 2006 | Since 2006 |
|---|---|---|
| Constitution Day | February 5 | 1st Monday of February |
| Benito Juarez's Birthday | March 21 | 3rd Monday of March |
| Mexican Revolution Day | November 20 | 3rd Monday of November |

In [ ]:
df_nth = report_nth_monday_holidays(df_holidays)

print(f'Holidays with an nth-Monday rule (since 2006): {len(df_nth)} occurrences')
display(
    df_nth.rename(columns={
        'holiday_name': 'Holiday',
        'date': 'Observed date',
        'weekday_name': 'Weekday',
        'labor_law_rule': 'LFT rule',
    })
)

## 7. Compare outliers vs known holidays

In [ ]:
df_match, stats = compare_outliers_holidays(df_outliers, df_holidays)
df_outliers_cmp = df_match
df_match_display = df_match.copy()

n_total = stats['n_total']
n_match = stats['n_match']
n_unknown = stats['n_unknown']

print('Outlier comparison against the holiday catalog')
print(f'Total outliers: {n_total}')
print(f'Match a known holiday: {n_match} ({100 * n_match / n_total:.0f}%)')
print(f'No catalog match: {n_unknown} ({100 * n_unknown / n_total:.0f}%)')

cols_display = ['date', 'dow_name', 'segment', 'distance', 'holiday_name']

print('\nKnown holidays detected as outliers')
display(
    df_match_display[df_match_display['is_known_holiday']]
    .sort_values('holiday_name')[cols_display]
 )

print('\nOutliers without a catalog match')
display(
    df_match_display[~df_match_display['is_known_holiday']]
    .sort_values('holiday_name')[cols_display]
 )

In [ ]:
detected_dates = set(pd.to_datetime(df_outliers_cmp['date']).dt.normalize())
all_dates_in_data = set(df_wide.index.normalize())

df_missed = find_holidays_not_detected(
    df_holidays,
    all_dates_in_data,
    detected_dates,
    WEEKDAY_NAMES,
 )
df_missed_display = df_missed.copy()

print(f'Known holidays present in the data but not detected as outliers: {len(df_missed)}')
if not df_missed.empty:
    display(df_missed_display.sort_values('holiday_name')[['holiday_name', 'dow_name', 'date']])

date_sets = get_date_sets(df_outliers_cmp, df_holidays, all_dates_in_data)
outlier_dates_set = date_sets['outlier_dates_set']
holiday_dates_set = date_sets['holiday_dates_set']
match_dates_set = date_sets['match_dates_set']
unknown_dates_set = date_sets['unknown_dates_set']
missed_dates_set = date_sets['missed_dates_set']

## 8. Visualization — profiles by weekday and outliers

In [ ]:
plot_profiles_by_segment_dow(
    df_wide, HOUR_COLS, SEGMENT_LABELS, WEEKDAY_NAMES,
    match_dates_set, unknown_dates_set, missed_dates_set, UNIQUE_ID,
    df_holidays=df_holidays_display,
)

Detected profiles use the same colors as the plotting helpers:

| Color | Meaning |
|---|---|
| Green | Detected outlier that matches a catalog holiday |
| Red | Detected outlier without a catalog match |
| Blue | Catalog holiday that was not flagged as an outlier |
| Black dashed line | Group centroid |
| Faint gray | Regular days |

## 8b. Hourly profiles by holiday

Each subplot overlays all available yearly profiles for one holiday present in the data.

| Color | Meaning |
|---|---|
| Green | Year detected as an outlier |
| Blue | Year present but not detected as an outlier |
| Black dashed line | Average holiday profile across years |

In [ ]:
holiday_groups = build_holiday_groups(df_holidays_display, df_wide.index)

plot_profiles_by_holiday(
    df_wide, holiday_groups, df_holidays_display, HOUR_COLS,
    match_dates_set, outlier_dates_set, UNIQUE_ID,
)

## 8c. Cluster atypical profiles

This view clusters the detected atypical profiles and compares each cluster centroid against weekday reference profiles.

In [ ]:
N_CLUSTERS = 3
CLUSTER_COLORS = [
    '#e6194b', '#3cb44b', '#4363d8', '#f58231',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45',
]

cluster_results = cluster_atypical_profiles(
    df_wide,
    match_dates_set,
    set(),
    outlier_dates_set,
    N_CLUSTERS,
    HOUR_COLS,
    df_holidays_display,
 )
df_atyp = cluster_results['df_atyp']
df_sim = cluster_results['df_sim']
df_sim_display = df_sim.copy()

WEEKDAY_REFERENCE_COLUMNS = ['r vs Mon', 'r vs Tue', 'r vs Wed', 'r vs Thu', 'r vs Fri']
WEEKEND_REFERENCE_COLUMNS = ['r vs Sat', 'r vs Sun']
format_dict = {
    column_name: '{:.3f}'
    for column_name in WEEKDAY_REFERENCE_COLUMNS + WEEKEND_REFERENCE_COLUMNS
}

print('Similarity between each cluster centroid and the weekday reference profiles')
display(
    df_sim_display.style
    .background_gradient(subset=WEEKDAY_REFERENCE_COLUMNS, cmap='Blues', vmin=0, vmax=1)
    .background_gradient(subset=WEEKEND_REFERENCE_COLUMNS, cmap='Greens', vmin=0, vmax=1)
    .format(format_dict)
 )

plot_cluster_atypical(cluster_results, df_atyp, HOUR_COLS, CLUSTER_COLORS, UNIQUE_ID)

## 8d. Holiday A/B validation

This section tests whether each holiday tends to behave more like a Saturday profile or a Sunday profile across its observed occurrences.

In [ ]:
AB_ALPHA = 0.10
AB_MIN_OCCURRENCES = 3

holiday_type_results = classify_holiday_weekend_type(
    df_wide=df_wide,
    df_holidays=df_holidays,
    outlier_dates_set=outlier_dates_set,
    hour_cols=HOUR_COLS,
    alpha=AB_ALPHA,
    min_occurrences=AB_MIN_OCCURRENCES,
 )

df_holiday_type_occ = holiday_type_results['occurrences_df'].copy()
df_holiday_type_summary = holiday_type_results['summary_df'].copy()
overall_ab_stats = holiday_type_results['overall_stats']

df_holiday_type_occ_display = df_holiday_type_occ.copy()
df_holiday_type_summary_display = df_holiday_type_summary.copy()

print('Holiday A/B validation')
print(f"Occurrences evaluated: {overall_ab_stats.get('n_occurrences', 0)}")
print(f"Holidays with data: {overall_ab_stats.get('n_holidays', 0)}")
print(f"Global Wilcoxon p-value: {overall_ab_stats.get('overall_wilcoxon_pvalue', float('nan')):.4f}")
print(f"Global sign-test p-value: {overall_ab_stats.get('overall_sign_test_pvalue', float('nan')):.4f}")
print(f"Decision alpha: {overall_ab_stats.get('alpha', AB_ALPHA):.2f}")

holiday_type_labels = {
    'A': 'A - Saturday-like',
    'B': 'B - Sunday-like',
    'Mixed': 'Mixed / unclear',
    'Insufficient': 'Insufficient data',
}

df_holiday_type_summary_display['holiday_type_label'] = (
    df_holiday_type_summary_display['holiday_type']
    .map(holiday_type_labels)
    .fillna(df_holiday_type_summary_display['holiday_type'])
 )

summary_columns = [
    'holiday_name',
    'holiday_type',
    'holiday_type_label',
    'n_occurrences',
    'sat_like_votes',
    'sun_like_votes',
    'mixed_votes',
    'mean_corr_sat',
    'mean_corr_sun',
    'median_delta_corr',
    'wilcoxon_pvalue',
    'sign_test_pvalue',
    'decision',
]

display(
    df_holiday_type_summary_display[summary_columns]
    .sort_values(['holiday_type', 'holiday_name'])
    .reset_index(drop=True)
 )

df_holiday_ab_list = (
    df_holiday_type_summary_display[
        df_holiday_type_summary_display['holiday_type'].isin(['A', 'B'])
    ][[
        'holiday_name',
        'holiday_type',
        'holiday_type_label',
        'n_occurrences',
        'sat_like_votes',
        'sun_like_votes',
        'median_delta_corr',
        'wilcoxon_pvalue',
        'sign_test_pvalue',
    ]]
    .sort_values(['holiday_type', 'holiday_name'])
    .reset_index(drop=True)
 )

print('\nOperational A/B list by holiday')
display(df_holiday_ab_list)

HOLIDAY_TYPE_MAP = dict(
    zip(df_holiday_ab_list['holiday_name'], df_holiday_ab_list['holiday_type'])
 )
print('HOLIDAY_TYPE_MAP =')
print(HOLIDAY_TYPE_MAP)

occurrence_columns = [
    'holiday_name',
    'date',
    'year',
    'corr_sat',
    'corr_sun',
    'dist_sat',
    'dist_sun',
    'delta_corr_sat_minus_sun',
    'occurrence_type',
]

display(
    df_holiday_type_occ_display[occurrence_columns]
    .sort_values(['holiday_name', 'date'])
    .reset_index(drop=True)
 )

## 9. Distance distribution with threshold

In [ ]:
plot_distance_distribution(df_dist, SEGMENT_LABELS, OUTLIER_PERCENTILE, UNIQUE_ID)

## 10. Final summary

In [ ]:
print_summary(
    UNIQUE_ID, df_wide, n_total, n_match, n_unknown,
    df_missed, df_outliers_cmp, OUTLIER_PERCENTILE, WEEKDAY_NAMES,
)